In [ ]:
import pandas as pd
df = pd.read_csv("data1C/data_per_day.csv")

feature_cols = df.columns.drop(["id", "date", "mood"])

df["num_nonzero"] = ((df[feature_cols] != 0) & (~df[feature_cols].isna())).sum(axis=1)

# delete rows where num_nonzero < 4 and mood is missing
df["is_fake"] = (df["num_nonzero"] < 4) & (df["mood"].isna())

df_clean = df[~df["is_fake"]].copy()

NameError: name 'pd' is not defined

In [ ]:
df_clean["date"] = pd.to_datetime(df_clean["date"])

# Create next-day target and next date within each user
df_clean = df_clean.sort_values(["id", "date"]).copy()
df_clean["mood_next_day"] = df_clean.groupby("id")["mood"].shift(-1)
df_clean["next_date"] = df_clean.groupby("id")["date"].shift(-1)

# Consecutive-day condition
is_next_day = (df_clean["next_date"] - df_clean["date"]).dt.days == 1

In [ ]:
# Option A: require mood today and next-day mood
df_model_A = df_clean[
    is_next_day &
    df_clean["mood"].notna() &
    df_clean["mood_next_day"].notna()
].copy()

In [ ]:
# Option B: allow missing mood today, but require some real activity today
df_model_B = df_clean[
    is_next_day &
    df_clean["mood_next_day"].notna() &
    (df_clean["num_nonzero"] >= 3)
].copy()

In [ ]:
# Save both
df_model_A.to_csv("data1C/model_option_A.csv", index=False)
df_model_B.to_csv("data1C/model_option_B.csv", index=False)